# Scratch: week of Oct 19-25
Exploration notebook. Data comes from the company app (Supabase); credentials load from `.env` in the project root.

In [1]:
import sys
from datetime import timedelta
sys.path.insert(0, "../src")
from reusables import *
from supabase_pull import load_waste, count_jobs, split_baseline_events, pull
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.float_format", "{:,.2f}".format)

# ---- Parameters -------------------------------------------------------------
#Reporting week#
WEEK_START  = "2026-10-19"      # Monday of the week
WEEK_LABEL  = "Week of Oct 19-25"
WEEK_JOBS   = 32           # None = count from the app
LOAD_START  = "2026-01-01"

#Previous week#
LAST_WEEK_START = "2026-10-12"
LAST_WEEK_END   = "2026-10-18"
LAST_WEEK_LABEL = "Week of Oct 12-18"
LAST_WEEK_JOBS  = 36       # None = count from the app

#Comparison#
SHOW_RUNNING_AVG = True    # adds an average-week column (average of the weeks compared)
# ---------------------------------------------------------------------------

week_start = pd.Timestamp(WEEK_START).date()
week_end   = week_start + timedelta(days=7)          # following Monday (not included)
last_start = pd.Timestamp(LAST_WEEK_START).date()
last_end   = pd.Timestamp(LAST_WEEK_END).date()

# All waste events up to the end of the week (app timestamps are UTC, so pull one extra day and filter on local date)
df = load_waste(LOAD_START, end=str(week_end + timedelta(days=1)))
df = df[df["incident_date"] < week_end].copy()
df["kind"] = df["kind"].replace({"Lost Item": "Item Search"})

# The Phase 1 baseline import carries its import date, not real dates, so keep it out of every week
imported_baseline, df = split_baseline_events(df)

this_week = df[df["incident_date"] >= week_start]
last_week = df[(df["incident_date"] >= last_start) & (df["incident_date"] <= last_end)]

if WEEK_JOBS is None:                                 # jobs with labor logged in the app that week
    WEEK_JOBS = count_jobs(str(week_start), str(week_end - timedelta(days=1)))
if LAST_WEEK_JOBS is None:
    LAST_WEEK_JOBS = count_jobs(str(last_start), str(last_end))

weeks        = {LAST_WEEK_LABEL: last_week, WEEK_LABEL: this_week}   # oldest first; the last one is the reporting week
jobs_by_week = {LAST_WEEK_LABEL: LAST_WEEK_JOBS, WEEK_LABEL: WEEK_JOBS}

checks = run_checks(df, {
    "missing date":   df["incident_date"].isna(),
    "zero cost":      df["total_cost"].le(0),
    "no job linked":  df["job_id"].isna() & df["kind"].isin(["Breakage", "Reclean", "Repeat Trip"]),
})
print(f"{len(df)} events loaded ({len(imported_baseline)} baseline-import events excluded)")
print(f"{LAST_WEEK_LABEL}: {len(last_week)} events, {LAST_WEEK_JOBS} jobs | {WEEK_LABEL}: {len(this_week)} events, {WEEK_JOBS} jobs")

17 events loaded (0 baseline-import events excluded)
Week of Oct 12-18: 8 events, 36 jobs | Week of Oct 19-25: 9 events, 32 jobs


## Data checks

In [2]:
checks

,rows_flagged,of_total,pct
no job linked,1,17,5.90
missing date,0,17,0.00
zero cost,0,17,0.00


## This week by type

In [3]:
per_100_jobs(this_week, WEEK_JOBS)

,events,total,events_per_100,cost_per_100
kind,,,,
Breakage,4,616.00,12.50,"1,925.00"
Item Search,5,180.95,15.60,565.47


## Previous week vs this week
Change is the reporting week against the previous week and against the average week (average of the weeks shown).

In [4]:
compare_weeks(weeks, jobs_by_week, running_avg=SHOW_RUNNING_AVG)

Week of Oct 12-18                           Week of Oct 19-25  \
                       Events Cost ($) Per 100 jobs ($)            Events   
kind                                                                        
Breakage                 5.00   136.50           379.17              4.00   
Item Search              2.00   378.00         1,050.00              5.00   
Repeat Trip              1.00   112.00           311.11              0.00   
Total                    8.00   626.50         1,740.28              9.00   

                                      Running avg                            \
            Cost ($) Per 100 jobs ($)      Events Cost ($) Per 100 jobs ($)   
kind                                                                          
Breakage      616.00         1,925.00        4.50   376.25         1,106.62   
Item Search   180.95           565.47        3.50   279.48           821.99   
Repeat Trip     0.00             0.00        0.50    56.00           164.71   
Total         796.95         2,490.47        8.50   711.72         2,093.31   

            Change vs Week of Oct 12-18         Change vs Running avg          
                              $ per 100       %             $ per 100       %  
kind                                                                           
Breakage                       1,545.83  408.00                818.38   74.00  
Item Search                     -484.53  -46.00               -256.52  -31.00  
Repeat Trip                     -311.11 -100.00               -164.71 -100.00  
Total                            750.19   43.00                397.16   19.00

## Event detail

In [5]:
this_week[["incident_date", "event_no", "kind", "job", "job_type", "title", "hours", "labor_cost", "other_cost", "total_cost"]]

,incident_date,event_no,kind,job,job_type,title,hours,labor_cost,other_cost,total_cost
8,2026-10-20,W-2026-0026,Item Search,6260 Lake Plantation Dr,water,Search,1.63,34.30,0.00,34.30
9,2026-10-20,W-2026-0027,Item Search,6260 Lake Plantation Dr,water,Search,2.92,61.25,0.00,61.25
10,2026-10-20,W-2026-0028,Item Search,6260 Lake Plantation Dr,water,Search,0.40,8.40,0.00,8.40
11,2026-10-21,W-2026-0032,Breakage,854 Summer Bay Dr,water,Broken glass,0.00,0.00,50.00,50.00
12,2026-10-21,W-2026-0033,Breakage,854 Summer Bay Dr,water,Dented white shelf,0.00,0.00,21.00,21.00
13,2026-10-21,W-2026-0024,Item Search,6260 Lake Plantation Dr,water,Search,1.83,38.50,0.00,38.50
14,2026-10-21,W-2026-0025,Item Search,6260 Lake Plantation Dr,water,Search,1.83,38.50,0.00,38.50
15,2026-10-22,W-2026-0023,Breakage,NaN,NaN,Furniture damaged,0.00,0.00,495.00,495.00
16,2026-10-23,W-2026-0029,Breakage,1322 Ingleside Ave,water,Broken glass,0.00,0.00,50.00,50.00


## Labor logged this week
`pull()` runs any read-only query and returns a DataFrame. Pass `snapshot="name"` to save a dated copy to `data/raw`.

In [6]:
labor = pull("""
    select l.work_date, l.stage, l.hours_worked, l.job_id, j.service_type
    from daily_labor l
    left join jobs j on j.id = l.job_id
    where l.work_date between %(s)s and %(e)s
""", {"s": str(week_start), "e": str(week_end - timedelta(days=1))})

labor.groupby(["work_date", "stage"])["hours_worked"].sum().unstack(fill_value=0)

stage,chemical_clean,demo,laundry,move_back,move_in,non_salvage,office,post_construction_clean,residential_clean,searching,structure_clean,warehouse,water_mitigation
work_date,,,,,,,,,,,,,
2026-10-19,0.00,0.00,0.00,39.75,28.78,0.00,10.07,0.00,37.63,0.00,13.98,0.00,7.85
2026-10-20,47.03,0.00,8.06,0.00,0.00,0.00,16.95,0.00,67.56,4.54,14.00,0.79,13.61
2026-10-21,6.79,0.00,8.34,0.00,7.54,8.77,19.77,39.00,27.33,3.68,12.80,0.00,13.74
2026-10-22,0.00,7.60,0.00,0.00,0.00,5.90,18.92,42.14,2.52,0.00,14.02,29.56,6.23
2026-10-23,0.00,0.00,8.41,48.65,0.00,0.00,8.84,39.21,0.00,0.00,12.47,0.01,3.36
2026-10-24,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.44


## Scratch

job count from week of 10/19

In [7]:
job_count = 32
print(f"Job Count for week of 10/19: {job_count}")

Job Count for week of 10/19: 32


Comparison of 10/19 week to 10/12 week